# 06.05 · Vector stores: in-memory → Chroma → production options

**Module 06 · Documents, Embeddings & RAG** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 06.02–06.04. |
| 🔑 Keys | none (no model calls in this lesson) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Use** the `VectorStore` interface: `add_documents(ids=...)`, `similarity_search(_with_score)`, `delete`, `as_retriever`.
2. **Persist** a Chroma index **outside git** (`data/vectorstores/`), reopen it without re-embedding, and upsert by id.
3. **Filter** by metadata (type, source, page) to scope a search, e.g. per product or per user.
4. **Update** an index when one document changes, and delete an index safely on Windows.
5. **Compare** production options: pgvector, Qdrant, Pinecone (and when Chroma is enough).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Semantic search: vector stores](https://docs.langchain.com/oss/python/langchain/knowledge-base) | `InMemoryVectorStore`, `add_documents`, similarity search, retrievers |
| [Chroma (LangChain)](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) | `Chroma(collection_name, embedding_function, persist_directory)`, filters, update/delete |
| [Chroma: configure collections](https://docs.trychroma.com/docs/collections/configure) | Distance space (`cosine`, `l2`, `ip`) and HNSW settings |
| [PGVector](https://docs.langchain.com/oss/python/integrations/vectorstores/pgvector), [Qdrant](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant), [Pinecone](https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone) | The production options surveyed in Part 6 |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langchain-text-splitters>=1.1.3,<2" "pypdf>=6.20.1,<7" "beautifulsoup4>=4.15.0,<5" "langchain-chroma>=1.1.0,<2" "fastembed>=0.9.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "langchain-chroma", "chromadb",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/06-rag/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out. Set these BEFORE importing chromadb/fastembed.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma: no usage telemetry.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: HF cache can't symlink without Developer Mode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")        # HF tokenizers' fork warning in Jupyter.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# tqdm (used by the model downloader) asks for ipywidgets to draw a progress bar; text output is fine.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
# Step 5: The course corpus: 24 Markdown files + 8 PDFs (23 pages) about fictional Nimbus weather stations.
import json

RAG_DIR = REPO_ROOT / "data" / "samples" / "rag"
CORPUS_DIR = RAG_DIR / "corpus"
GOLDEN = [json.loads(line) for line in (RAG_DIR / "golden-set.jsonl").read_text(encoding="utf-8").splitlines()]
print("corpus files:", sum(1 for p in CORPUS_DIR.rglob("*") if p.is_file()), "| golden questions:", len(GOLDEN))

> 🌐 First run downloads a small model from Hugging Face (no key, no tokens), cached afterwards. Offline, blocked, or `[embeddings]` not installed? The cell prints a ⚠️ and the lesson continues with the offline `HashingEmbeddings`.

In [ ]:
# Step 6 (🌐 first run only): chunks + the best available embeddings (fastembed bge-small, else HashingEmbeddings).
from mastery.rag import default_embeddings, embedding_label, load_corpus, split_documents

chunks = split_documents(load_corpus(CORPUS_DIR), "markdown", 800, 100)
embeddings = default_embeddings()
EMB = embedding_label(embeddings)
print(len(chunks), "chunks | embeddings:", EMB)

**You should now see** `73 chunks | embeddings: BAAI/bge-small-en-v1.5` (or `HashingEmbeddings-1024` with a ⚠️ warning if the local model isn't available).

---
## Part 1 · `InMemoryVectorStore`: the simplest store

Lives in RAM, disappears when the kernel stops. Perfect for notebooks, tests and small corpora (a few thousand chunks).

In [ ]:
# 🧪 Offline: add with ids, search with scores, filter with a function.
from langchain_core.vectorstores import InMemoryVectorStore

memory_store = InMemoryVectorStore(embeddings)
ids = memory_store.add_documents(chunks, ids=[c.metadata["chunk_id"] for c in chunks])
print("stored:", len(memory_store.store), "| first id:", ids[0])

question = "How long is the warranty of the Nimbus Marine?"
for doc, score in memory_store.similarity_search_with_score(question, k=3):
    print(f"{score:.3f}  {doc.metadata['source']:<32} {doc.metadata.get('section', '')}")

only_pdfs = memory_store.similarity_search(question, k=2, filter=lambda d: d.metadata["type"] == "pdf")
print("PDF-only:", [d.metadata["source"] for d in only_pdfs])

**You should now see** `stored: 73`, three hits from `products/nimbus-marine.md` and the Marine manual, with their `section` (Markdown chunks) and cosine **similarity** scores (higher = closer), and two PDF-only hits. In-memory filters are plain Python functions.

`memory_store.dump(path)` / `InMemoryVectorStore.load(path, embeddings)` save it as JSON, fine for tiny demos; for anything real, use a database.

---
## Part 2 · Chroma, persisted outside git

[Chroma](https://docs.trychroma.com) runs **inside your Python process** (no server, no Docker) and stores everything in a folder (SQLite + index files). We put it in `data/vectorstores/`, which `.gitignore` excludes. Two settings matter:

- **`collection_name`**: name it after the corpus **and the embedding model**, so you never mix vectors from two models.
- **distance**: Chroma's default is `l2`; we ask for **cosine** (`collection_configuration={"hnsw": {"space": "cosine"}}`), the usual choice for text embeddings.

In [ ]:
# 🧪 Offline (local files): create a persisted collection with the plain langchain-chroma API.
import chromadb
from langchain_chroma import Chroma

from mastery.rag import folder_size, vectorstore_dir

COLLECTION = "nimbus-" + "".join(ch if ch.isalnum() else "-" for ch in EMB.split("/")[-1].lower())[:40]
PERSIST = vectorstore_dir(COLLECTION)          # <repo>/data/vectorstores/<collection>
chroma = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(PERSIST),
    client_settings=chromadb.config.Settings(anonymized_telemetry=False, is_persistent=True, persist_directory=str(PERSIST)),
    collection_configuration={"hnsw": {"space": "cosine"}},
)
clean = [c.model_copy(update={"metadata": {k: v for k, v in c.metadata.items() if v is not None}}) for c in chunks]
chroma.add_documents(clean, ids=[c.metadata["chunk_id"] for c in clean])
print("collection:", COLLECTION, "| count:", chroma._collection.count(), "| folder:", PERSIST.relative_to(REPO_ROOT).as_posix(),
      "|", folder_size(PERSIST))

**You should now see** `collection: nimbus-bge-small-en-v1-5 | count: 73 | folder: data/vectorstores/nimbus-bge-small-en-v1-5 | ~1–2 MB`.

Chroma accepts only `str`, `int`, `float` and `bool` metadata values: we dropped `None` values (Markdown chunks have no `page`). `mastery.rag.build_index(kind="chroma")` does all of this in one call.

**Run the previous cell again**: the count stays 73. Adding with the **same ids** is an *upsert* (update or insert), which is why stable `chunk_id`s (06.03) matter. Random ids would give you 146, then 219...

In [ ]:
# 🧪 Offline: reopen the collection in a NEW object (like a new kernel would): no re-embedding.
from mastery.rag import open_chroma, retrieve

reopened = open_chroma(COLLECTION, embeddings)
print("reopened count:", reopened._collection.count())
for doc, distance in reopened.similarity_search_with_score(question, k=3):
    print(f"distance={distance:.3f}  {doc.metadata['source']}")
print("same, as similarity:", [(d.metadata["source"], d.metadata["score"]) for d in retrieve(reopened, question, k=1)])

**You should now see** `reopened count: 73`, then three hits with **distances** (lower = closer: Chroma returns cosine *distance* = 1 − similarity), and the same top hit from `retrieve()`, which converts distances to similarities so both stores report scores the same way.

Reopen with the **same embedding model**. Chroma doesn't check: a different model gives silently wrong results (the collection name is your guard).

---
## Part 3 · Metadata filters

Filters run **inside** the store, before or during the vector search. Use them to scope by product, document type, language, date, or **user / tenant** (so one customer never sees another's documents).

In [ ]:
# 🧪 Offline: Chroma "where" filters ($eq, $in, $gte, $and ...).
q = "What should I do about errors on my station?"
filters = {
    "no filter": None,
    "PDFs only": {"type": "pdf"},
    "Pro manual, page ≥ 3": {"$and": [{"source": "manuals/nimbus-pro-manual.pdf"}, {"page": {"$gte": 3}}]},
    "support FAQs": {"source": {"$in": ["support/faq-wifi.md", "support/faq-firmware.md", "support/faq-battery.md"]}},
}
for label, where in filters.items():
    hits = retrieve(reopened, q, k=3, filter=where)
    print(f"{label:<22}", [(d.metadata["source"].split("/")[-1], d.metadata.get("page")) for d in hits])

retriever = reopened.as_retriever(search_kwargs={"k": 2, "filter": {"type": "markdown"}})
print("as_retriever:", [d.metadata["source"] for d in retriever.invoke(q)])

**You should now see** four rows: unfiltered hits from anywhere, PDF-only hits, only pages 3–4 of the Pro manual, only the three FAQs; then two Markdown sources from `as_retriever`. A **retriever** is the standard "query in, documents out" interface (a `Runnable`), handy in chains; `retrieve()` is the same thing with scores.

---
## Part 4 · When a document changes

Indexes go stale. When a file changes: delete its old chunks, add the new ones. Our ids start with the source path, and Chroma can find chunks by metadata.

In [ ]:
# 🧪 Offline: re-index one changed file (a fake edit to the returns policy).
from langchain_core.documents import Document

from mastery.rag import split_documents

SOURCE = "policies/returns.md"
old_ids = reopened.get(where={"source": SOURCE})["ids"]
reopened.delete(ids=old_ids)
edited = Document("# Returns policy\n\nYou can return any Orbita product within 45 days (changed for this demo).",
                  metadata={"source": SOURCE, "type": "markdown", "title": "Returns policy"})
new_chunks = split_documents([edited], "markdown", 800, 100)
reopened.add_documents(new_chunks, ids=[c.metadata["chunk_id"] for c in new_chunks])
print("removed:", old_ids, "→ added:", [c.metadata["chunk_id"] for c in new_chunks])
print("top hit now:", retrieve(reopened, "how many days to return a product", k=1)[0].page_content[-60:])

# Put the real file back so later notebooks see the original policy.
reopened.delete(ids=[c.metadata["chunk_id"] for c in new_chunks])
original = [c for c in clean if c.metadata["source"] == SOURCE]
reopened.add_documents(original, ids=[c.metadata["chunk_id"] for c in original])
print("restored count:", reopened._collection.count())

**You should now see** the removed id(s) (`policies/returns.md#p0-c0`: the file is short, one chunk), the added id, `top hit now: ... within 45 days (changed for this demo).`, then `restored count: 73`.

### What just happened?

Stable ids made the update surgical: no full rebuild, no duplicates. In production a small job compares file hashes or modification times and re-indexes only what changed; LangChain's indexing API automates the bookkeeping: `index()` with `cleanup="incremental"` skips unchanged chunks and **deletes the orphans** that a shortened or removed document leaves behind, which plain upserts never do. `index()` and an in-memory `RecordManager` ship in `langchain_core.indexing`; the persistent `SQLRecordManager` lives in `langchain-classic` (the v1 release notes list "the indexing API" there). One difference to know: `index()` picks its own ids (derived from a hash of each chunk's content and metadata), not your `chunk_id`.

---
## Part 5 · Jupyter skill: data hygiene

Know what your notebook writes, and make sure none of it reaches git.

In [ ]:
# 🧪 Offline: what did Module 06 write so far, and where?
for folder in [REPO_ROOT / "data" / "vectorstores", REPO_ROOT / "data" / "cache"]:
    if folder.exists():
        for sub in sorted(p for p in folder.iterdir() if p.is_dir()):
            print(f"{sub.relative_to(REPO_ROOT).as_posix():<55} {folder_size(sub)}")
gitignore = (REPO_ROOT / ".gitignore").read_text(encoding="utf-8").splitlines()
print("\nignored by .gitignore:", [rule for rule in gitignore if rule in ("data/vectorstores/", "data/cache/")])

**You should now see** your collection folder under `data/vectorstores/` (and `data/cache/embeddings` if you ran 06.04) with their sizes (a `scratch-demo` folder appears after you've run the next cell once), and both rules found in `.gitignore`.

Confirm with git itself, in **Git Bash** at the repo root:

```bash
git check-ignore -v data/vectorstores/ data/cache/   # prints the .gitignore line that matches each path
git status --short                                   # nothing under data/vectorstores or data/cache should appear
```

Habits: one folder per index, named after corpus + model; delete indexes you don't need (`reset_chroma(store)`); keep sample documents small and fictional; never commit real customer documents.

**Windows:** an open Chroma client keeps its files locked. Deleting the folder in Explorer (or `shutil.rmtree`) while the kernel runs fails with `PermissionError: [WinError 32]`. Delete the **collection** through the store, or restart the kernel first.

In [ ]:
# 🧪 Offline: a throwaway collection, created and deleted safely (the course index stays).
from mastery.rag import build_index, reset_chroma

scratch = build_index(chunks[:5], embeddings, kind="chroma", collection="scratch-demo")
print("scratch count:", scratch._collection.count())
reset_chroma(scratch)                         # deletes the collection through the open client: no file-lock issue
print("scratch deleted. Course index still has:", reopened._collection.count())

**You should now see** `scratch count: 5`, then the deletion message and `73`. (The empty `data/vectorstores/scratch-demo/` folder can be removed after a kernel restart.)

---
## Part 6 · Production options

| Store | Runs | Cost | Strengths | Consider when |
|---|---|---|---|---|
| **InMemory** | Your process | Free | Zero setup | Tests, notebooks, < ~10k chunks |
| **Chroma** (`langchain-chroma` 1.1.0, chromadb 1.5) | Your process (or Chroma server / cloud) | Free locally | Simple, persistent, filters | Prototypes, single-machine apps, this course |
| **pgvector** (`langchain-postgres`) | PostgreSQL extension (Docker locally) | Free (self-hosted) / your Postgres bill | SQL + vectors in one DB, transactions, joins, backups you already have | You already run Postgres; need filters on relational data |
| **Qdrant** (`langchain-qdrant`) | Server (Docker), embedded, or cloud | Free self-hosted; cloud free tier | Fast filtering, hybrid (sparse + dense) search | Larger corpora, heavy filtering |
| **Pinecone** (`langchain-pinecone`) | Fully managed cloud | Free starter tier, then usage-based | No ops, scales | You don't want to run a database |

All share the `VectorStore` interface, so moving later is mostly a constructor change, **plus re-embedding** if you change the model. Project 06's stretch goal runs pgvector in Docker.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `InvalidArgumentError: ... Expected a name containing 3-512 characters` | Collection name too short or with spaces | 3+ characters from `[a-zA-Z0-9._-]` |
| `ValueError: Expected metadata value to be a str, int, float or bool, got None` | `None` or list metadata | Drop `None`, join lists into strings (as above) |
| Count doubles on every run | `add_documents` without ids | Pass stable ids (upsert) |
| `PermissionError: [WinError 32]` deleting `data/vectorstores/...` | Chroma files still open | `reset_chroma(store)`, or restart the kernel before deleting |
| Scores look "reversed" | Chroma returns distance (lower = better) | Use `retrieve()` or `1 - distance` |
| Results are nonsense after reopening | Reopened with another embedding model | Same model; name collections after the model |
| Telemetry messages | Chroma's anonymous telemetry | Step 4 sets `ANONYMIZED_TELEMETRY=False`; we also pass `anonymized_telemetry=False` |

## ⚠️ Production notes

> ⚠️ **Production note: filter by tenant inside the store.**
> For multi-user apps put `user_id`/`tenant_id` in every chunk's metadata and **always** pass it as a filter on the server side. Filtering after retrieval leaks other users' chunks into the top-k and the prompt.

> ⚠️ **Production note: indexes need backups and rebuilds.**
> Keep the source documents and the indexing code as the source of truth; you should be able to rebuild any index from scratch with one command.

## 🏋️ Exercises

1. **(Easy)** Count the chunks per `type` in the Chroma collection with `reopened.get(where={"type": "pdf"})`.
2. **(Medium)** Search "battery" restricted to the `compliance/` PDFs only. (Hint: `$in` with both file names.)
3. **(Medium)** Run the Part 2 cell twice, then once with `ids=None`. What's the count? How do you clean up?
4. **(Stretch)** Write `reindex_file(store, path)` that loads one file, deletes its old chunks by `source` and adds the new ones. Test it on a temp copy of a Markdown file.

<details>
<summary><b>Show solutions</b></summary>

**1.** `len(reopened.get(where={"type": "pdf"})["ids"])` → 23; markdown → 50.

**2.** `retrieve(reopened, "battery", k=3, filter={"source": {"$in": ["compliance/battery-handling.pdf", "compliance/safety-and-compliance.pdf"]}})`.

**3.** Twice with ids → 73. With `ids=None`, Chroma generates random ids → 146. Clean up: `reset_chroma(chroma)` and rebuild, or delete the random-id rows (`get()` the ids that don't contain `#p`).

**4.** `docs = load_markdown(path, root=CORPUS_DIR)` (or `load_pdf`), `old = store.get(where={"source": docs[0].metadata["source"]})["ids"]`, `store.delete(ids=old)` if `old`, `new = split_documents(docs, "markdown", 800, 100)`, `store.add_documents(new, ids=[c.metadata["chunk_id"] for c in new])`.
</details>

## ✅ Checkpoint

- [ ] I used `InMemoryVectorStore` and Chroma through the same interface.
- [ ] My Chroma index lives in `data/vectorstores/` (git-ignored), named after corpus + model, and reopens without re-embedding.
- [ ] I can filter by metadata and update one document's chunks.
- [ ] I know how to delete an index on Windows without file-lock errors.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

What distance does a new Chroma collection use by default today, and how do you choose another (`collection_configuration` vs the older `collection_metadata={"hnsw:space": ...}`)? See [Chroma: configure](https://docs.trychroma.com/docs/collections/configure). Log the answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Same interface everywhere: `add_documents(ids)`, `similarity_search(_with_score)`, `delete`, `as_retriever`.
- Chroma persists in-process to a git-ignored folder; stable ids make re-runs upserts.
- Filters scope searches (and protect tenants); scores vs distances need care.
- pgvector, Qdrant or Pinecone when you outgrow one machine; re-embed when you change models.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 06.06 · 2-step RAG with citations** → [`06-06-two-step-rag.ipynb`](06-06-two-step-rag.ipynb).

[Module 06 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)